# mcp_call_log Lakebase → UC Delta sync

Idempotent incremental copy of new rows from the Lakebase OLTP
`mcp_call_log` table into the UC Delta `mcp_call_log` table.

Strategy:
  1. Read max(created_at) from the UC Delta target as the watermark
     (NULL → epoch on the first run).
  2. SELECT every Lakebase row WHERE created_at > watermark, ordered
     ASCENDING by created_at so a partial run still leaves the next
     run with a sane watermark.
  3. Write the batch to Delta in append mode.

Failure mode: on Lakebase or Delta error, raise so the job fails and
Databricks Jobs retries per its policy. The next successful run picks
up where the previous one left off (watermark is in the Delta target,
not in the job).

In [ ]:
dbutils.widgets.text('catalog_use', '')
dbutils.widgets.text('schema_use', '')
dbutils.widgets.text('lakebase_project_id', '')
dbutils.widgets.text('lakebase_database', 'databricks_postgres')
dbutils.widgets.text('batch_size', '5000')

catalog = dbutils.widgets.get('catalog_use')
schema = dbutils.widgets.get('schema_use')
project_id = dbutils.widgets.get('lakebase_project_id')
database = dbutils.widgets.get('lakebase_database')
batch_size = int(dbutils.widgets.get('batch_size'))

target_table = f'{catalog}.{schema}.mcp_call_log'
print(f'target: {target_table}')
print(f'lakebase project: {project_id} / database: {database}')
print(f'batch_size: {batch_size}')

## 1. Resolve the watermark from the UC Delta target

In [ ]:
from datetime import datetime, timezone

EPOCH = datetime(1970, 1, 1, tzinfo=timezone.utc)

wm_row = spark.sql(f'SELECT MAX(created_at) AS wm FROM {target_table}').collect()
watermark = wm_row[0]['wm'] if wm_row and wm_row[0]['wm'] is not None else EPOCH
print(f'watermark: {watermark.isoformat()}')

## 2. Connect to Lakebase via OAuth-rotated Postgres

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.database import DatabaseInstanceRole
import psycopg

w = WorkspaceClient()
branch_name = f'projects/{project_id}/branches/production'
endpoints = w.postgres.list_endpoints(branch_name).endpoints or []
endpoint = endpoints[0]
host = endpoint.config.host
port = endpoint.config.port or 5432

cred = w.database_credentials.generate_database_credential(
    request_id=f'mcp-call-log-sync-{datetime.now(timezone.utc).timestamp():.0f}',
    instance_names=[branch_name],
)
token = cred.token
username = w.current_user.me().user_name
print(f'connecting as {username} to {host}:{port}/{database}')

In [ ]:
conn = psycopg.connect(
    host=host, port=port, user=username, password=token,
    dbname=database, sslmode='require',
)
conn.autocommit = True
print('connected')

## 3. Read incremental batch from Lakebase

In [ ]:
import json

rows = []
with conn.cursor() as cur:
    cur.execute(
        '''
        SELECT call_id, request_id, tenant_id, user_id, tool_name,
               tool_input::text AS tool_input,
               tool_output_summary, sage_calls_made, latency_ms,
               status, error_message, created_at
          FROM mcp_call_log
         WHERE created_at > %s
         ORDER BY created_at ASC
         LIMIT %s
        ''',
        (watermark, batch_size),
    )
    for r in cur:
        rows.append({
            'call_id': r[0],
            'request_id': r[1],
            'tenant_id': r[2],
            'user_id': r[3],
            'tool_name': r[4],
            # tool_input is captured as JSON text — parsed and stored as VARIANT.
            'tool_input': json.loads(r[5]) if r[5] else None,
            'tool_output_summary': r[6],
            'sage_calls_made': r[7],
            'latency_ms': r[8],
            'status': r[9],
            'error_message': r[10],
            'created_at': r[11],
        })
print(f'fetched {len(rows)} new rows')

## 4. Append to UC Delta

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, LongType,
    TimestampType, VariantType,
)

if not rows:
    print('no new rows; nothing to write')
    dbutils.jobs.taskValues.set(key='rows_synced', value=0)
else:
    schema_def = StructType([
        StructField('call_id', StringType(), False),
        StructField('request_id', StringType(), True),
        StructField('tenant_id', StringType(), True),
        StructField('user_id', StringType(), True),
        StructField('tool_name', StringType(), False),
        StructField('tool_input', StringType(), True),  # serialized; cast to VARIANT below
        StructField('tool_output_summary', StringType(), True),
        StructField('sage_calls_made', IntegerType(), True),
        StructField('latency_ms', LongType(), True),
        StructField('status', StringType(), False),
        StructField('error_message', StringType(), True),
        StructField('created_at', TimestampType(), False),
    ])
    pre = [
        {**r, 'tool_input': json.dumps(r['tool_input']) if r['tool_input'] is not None else None}
        for r in rows
    ]
    df = (spark.createDataFrame(pre, schema_def)
        .withColumn('tool_input', F.expr('parse_json(tool_input)')))
    df.write.mode('append').saveAsTable(target_table)
    dbutils.jobs.taskValues.set(key='rows_synced', value=len(rows))
    print(f'appended {len(rows)} rows to {target_table}')

In [ ]:
conn.close()